read the london attarctions clustered

In [41]:
from pathlib import Path
import pandas as pd

RECOMMENDATION_DIR = Path.cwd()
PROJECT_ROOT = RECOMMENDATION_DIR.parent

DATA_FILE = (
    PROJECT_ROOT
    / "rag-pipeline"
    / "data"
    / "london_clustered_attractions.json"
)

print(DATA_FILE)

df = pd.read_json(DATA_FILE)
df.head()

/Users/efe/UNI/project/sightseer/rag-pipeline/data/london_clustered_attractions.json


,wikidata_id,name,category,description,latitude,longitude,image_url,sitelinks,summary,themes,interest_scores,recommended_visit_time,estimated_visit_mins,indoor,family_friendly,price_level,borough_name,cluster_id,cluster_label
0,Q6373,British Museum,museum,"national museum in London, United Kingdom",51.519444,-0.126944,http://commons.wikimedia.org/wiki/Special:File...,108,The British Museum is a world-renowned museum ...,"[history, art, culture, ancient history]","{'history': 5, 'art': 5, 'architecture': 4, 'n...",morning,120,True,True,free,Camden,18,"Camden and Marylebone — Literature, Art and Cu..."
1,Q62378,Tower of London,museum,"castle in central London, United Kingdom",51.508200,-0.076198,http://commons.wikimedia.org/wiki/Special:File...,88,The Tower of London is a historic castle and m...,"[history, architecture, royalty, military hist...","{'history': 5, 'art': 3, 'architecture': 5, 'n...",morning,120,True,True,££,Tower Hamlets,22,"City, Shoreditch and Bankside — History, Ident..."
2,Q192988,Royal Observatory,museum,"observatory in Greenwich, London, UK",51.477833,-0.001389,http://commons.wikimedia.org/wiki/Special:File...,61,"The Royal Observatory in Greenwich, London, is...","[science, history, technology]","{'history': 4, 'art': 2, 'architecture': 3, 'n...",morning,60,True,True,£,Greenwich,20,"Greenwich — Astronomy, Maritime and Ancient Hi..."
3,Q674773,Science Museum,museum,"science museum in London, United Kingdom",51.497500,-0.174722,http://commons.wikimedia.org/wiki/Special:File...,47,The Science Museum in London is a world-renown...,"[science, technology, history, education]","{'history': 4, 'art': 2, 'architecture': 2, 'n...",morning,120,True,True,£,Kensington and Chelsea,21,"Kensington — Science, Royalty and Art"
4,Q1990172,Sherlock Holmes Museum,museum,"museum in London, England",51.523694,-0.161000,http://commons.wikimedia.org/wiki/Special:File...,28,"The Sherlock Holmes Museum, located at 221B Ba...","[literature, history, entertainment]","{'history': 3, 'art': 2, 'architecture': 1, 'n...",morning,45,True,True,£,Westminster,18,"Camden and Marylebone — Literature, Art and Cu..."


create some mock users

In [42]:
MOCK_USERS = {
    "history_lover": {
        "history": 5,
        "art": 3,
        "architecture": 4,
        "nature": 2,
        "science": 2,
        "food": 1,
        "entertainment": 2,
        "shopping": 1,
        "views": 3,
        "family": 1,
    },
    "art_and_culture": {
        "history": 3,
        "art": 5,
        "architecture": 4,
        "nature": 2,
        "science": 2,
        "food": 2,
        "entertainment": 4,
        "shopping": 3,
        "views": 2,
        "family": 1,
    },
    "family_day_out": {
        "history": 3,
        "art": 2,
        "architecture": 2,
        "nature": 4,
        "science": 4,
        "food": 3,
        "entertainment": 5,
        "shopping": 2,
        "views": 3,
        "family": 5,
    },
    "nature_and_views": {
        "history": 2,
        "art": 2,
        "architecture": 3,
        "nature": 5,
        "science": 2,
        "food": 2,
        "entertainment": 3,
        "shopping": 1,
        "views": 5,
        "family": 3,
    },
    "food_shopping_and_fun": {
        "history": 1,
        "art": 2,
        "architecture": 2,
        "nature": 2,
        "science": 1,
        "food": 5,
        "entertainment": 5,
        "shopping": 5,
        "views": 3,
        "family": 2,
    },
}

INTERESTS = [
    "history",
    "art",
    "architecture",
    "nature",
    "science",
    "food",
    "entertainment",
    "shopping",
    "views",
    "family",
]

lets just test and try to get it to work for history lover

food shopping and fun doesnt have very high mathces as sightseer doesnt contain any food/shopping spots, but this can be imporved on later

In [43]:
import numpy as np

user_profile = MOCK_USERS["art_and_culture"]

user_vector = np.array(
    [user_profile[interest] for interest in INTERESTS],
    dtype=float,
)

user_vector

array([3., 5., 4., 2., 2., 2., 4., 3., 2., 1.])

need to convert the interest scores of each attarction into a vector as well

In [44]:
attraction_vectors = np.array([
    [scores[interest] for interest in INTERESTS]
    for scores in df["interest_scores"]
], dtype=float)

attraction_vectors.shape

(358, 10)

In [45]:
distances = np.linalg.norm(
    attraction_vectors - user_vector,
    axis=1,
)

recommendations = df.copy()
recommendations["recommendation_distance"] = distances

recommendations = recommendations.sort_values(
    "recommendation_distance",
    ascending=True,
)

recommendations[
    [
        "name",
        "cluster_label",
        "interest_scores",
        "recommendation_distance",
    ]
].head(10)

,name,cluster_label,interest_scores,recommendation_distance
55,Peckham Platform,"Southwark — Contemporary Art, Fashion and Prin...","{'history': 2, 'art': 5, 'architecture': 3, 'n...",3.464102
98,South London Gallery,"Southwark — Contemporary Art, Fashion and Prin...","{'history': 2, 'art': 5, 'architecture': 3, 'n...",3.605551
108,198 Contemporary Arts and Learning,"South London — Art, Nature and Science","{'history': 2, 'art': 5, 'architecture': 3, 'n...",3.605551
20,Fashion and Textile Museum,"Southwark — Contemporary Art, Fashion and Prin...","{'history': 3, 'art': 5, 'architecture': 2, 'n...",3.605551
77,Tate Modern,"City, Shoreditch and Bankside — History, Ident...","{'history': 2, 'art': 5, 'architecture': 4, 'n...",3.605551
111,Clore Gallery,"Westminster — Royalty, Art and Heritage","{'history': 2, 'art': 5, 'architecture': 4, 'n...",3.605551
88,Hayward Gallery,"Westminster — Royalty, Art and Heritage","{'history': 2, 'art': 5, 'architecture': 4, 'n...",3.605551
117,12 Duke Street,"Westminster — Royalty, Art and Heritage","{'history': 3, 'art': 5, 'architecture': 2, 'n...",3.741657
112,Gilbert & George Center,"City, Shoreditch and Bankside — History, Ident...","{'history': 2, 'art': 5, 'architecture': 2, 'n...",3.872983
89,Whitechapel Gallery,"City, Shoreditch and Bankside — History, Ident...","{'history': 3, 'art': 5, 'architecture': 2, 'n...",3.872983


Turning distance into a user-friendly score
A distance is useful internally, but displaying “distance = 4.47” is not intuitive. Since every category ranges from 1 to 5, the maximum possible distance across ten categories is: sqrt(10 * s(5-1)**2)


Therefore it should be converted into a 0–100 match percentage:

In [46]:
MAX_DISTANCE = np.sqrt(len(INTERESTS) * (5 - 1) ** 2)

recommendations["match_score"] = ((
    1 - recommendations["recommendation_distance"] / MAX_DISTANCE
) * 100).round(2)

recommendations[
    ["name", "cluster_label", "recommendation_distance", "match_score"]
].head(10)

,name,cluster_label,recommendation_distance,match_score
55,Peckham Platform,"Southwark — Contemporary Art, Fashion and Prin...",3.464102,72.61
98,South London Gallery,"Southwark — Contemporary Art, Fashion and Prin...",3.605551,71.50
108,198 Contemporary Arts and Learning,"South London — Art, Nature and Science",3.605551,71.50
20,Fashion and Textile Museum,"Southwark — Contemporary Art, Fashion and Prin...",3.605551,71.50
77,Tate Modern,"City, Shoreditch and Bankside — History, Ident...",3.605551,71.50
111,Clore Gallery,"Westminster — Royalty, Art and Heritage",3.605551,71.50
88,Hayward Gallery,"Westminster — Royalty, Art and Heritage",3.605551,71.50
117,12 Duke Street,"Westminster — Royalty, Art and Heritage",3.741657,70.42
112,Gilbert & George Center,"City, Shoreditch and Bankside — History, Ident...",3.872983,69.38
89,Whitechapel Gallery,"City, Shoreditch and Bankside — History, Ident...",3.872983,69.38


In [47]:
recommendations = recommendations.sort_values(
    ["cluster_id", "match_score"],
    ascending=[True, False],
)

# getting the top 3 attractions per cluster based on match score
top_attractions_per_cluster = (
    recommendations
    .groupby("cluster_id", group_keys=False)
    .head(3)
    .copy()
)

top_attractions_per_cluster[
    [
        "cluster_id",
        "cluster_label",
        "name",
        "match_score",
    ]
]

,cluster_id,cluster_label,name,match_score
108,0,"South London — Art, Nature and Science",198 Contemporary Arts and Learning,71.50
56,0,"South London — Art, Nature and Science",House of Dreams Museum,68.38
86,0,"South London — Art, Nature and Science",Dulwich Picture Gallery,65.54
111,1,"Westminster — Royalty, Art and Heritage",Clore Gallery,71.50
88,1,"Westminster — Royalty, Art and Heritage",Hayward Gallery,71.50
...,...,...,...,...
203,26,Hampstead and Highgate — Cemetery Heritage and...,Monument To Samuel Sanders Teulon In Highgate ...,62.09
204,26,Hampstead and Highgate — Cemetery Heritage and...,Monument To Thomas Mears In Highgate (Western)...,62.09
37,27,Bromley — Roman Archaeology and Historic Estates,Crofton Roman Villa,58.92
232,27,Bromley — Roman Archaeology and Historic Estates,Sheffield Monument (In Churchyard Of Parish Ch...,55.28


In [48]:
cluster_rankings = (
    top_attractions_per_cluster
    .groupby(
        ["cluster_id", "cluster_label"],
        as_index=False,
    )
    .agg(
        cluster_match_score=("match_score", "mean"),
        recommended_attractions=("name", list),
    )
)

cluster_rankings["cluster_match_score"] = (
    cluster_rankings["cluster_match_score"].round(2)
)

cluster_rankings = cluster_rankings.sort_values(
    "cluster_match_score",
    ascending=False,
).reset_index(drop=True)

cluster_rankings.head(10)

,cluster_id,cluster_label,cluster_match_score,recommended_attractions
0,19,"Southwark — Contemporary Art, Fashion and Prin...",71.87,"[Peckham Platform, South London Gallery, Fashi..."
1,1,"Westminster — Royalty, Art and Heritage",71.14,"[Clore Gallery, Hayward Gallery, 12 Duke Street]"
2,22,"City, Shoreditch and Bankside — History, Ident...",70.09,"[Tate Modern, Gilbert & George Center, Whitech..."
3,0,"South London — Art, Nature and Science",68.47,"[198 Contemporary Arts and Learning, House of ..."
4,21,"Kensington — Science, Royalty and Art",67.74,"[Leighton House, Serpentine Galleries, Victori..."
5,18,"Camden and Marylebone — Literature, Art and Cu...",67.41,"[Pushkin House, Wallace Collection, October Ga..."
6,8,"East London — Curiosities, Craft and Social Hi...",66.77,"[Cell Project Space, The Viktor Wynd Museum of..."
7,16,"Richmond — Music, Art and Landscapes",63.31,"[Orleans House Gallery, Eel Pie Island Museum,..."
8,26,Hampstead and Highgate — Cemetery Heritage and...,62.37,[Monument To The Emden Family In Highgate (Wes...
9,11,"Hackney — Art, Sport and Cemetery Heritage",61.01,"[Clowns Gallery-Museum, Arsenal Football Club ..."


In [49]:
cluster_rankings.insert(
    0,
    "rank",
    range(1, len(cluster_rankings) + 1),
)

cluster_rankings.head()

,rank,cluster_id,cluster_label,cluster_match_score,recommended_attractions
0,1,19,"Southwark — Contemporary Art, Fashion and Prin...",71.87,"[Peckham Platform, South London Gallery, Fashi..."
1,2,1,"Westminster — Royalty, Art and Heritage",71.14,"[Clore Gallery, Hayward Gallery, 12 Duke Street]"
2,3,22,"City, Shoreditch and Bankside — History, Ident...",70.09,"[Tate Modern, Gilbert & George Center, Whitech..."
3,4,0,"South London — Art, Nature and Science",68.47,"[198 Contemporary Arts and Learning, House of ..."
4,5,21,"Kensington — Science, Royalty and Art",67.74,"[Leighton House, Serpentine Galleries, Victori..."
